# Tugas Besar 2 IF3270 — CNN Image Classification
Dataset: Intel Image Classification (~25.000 gambar, 6 kelas)

## 0. Setup & Import

In [ ]:
import sys, os, glob
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from src.cnn.train import (
    load_dataset, build_cnn_model, build_locally_connected_model,
    run_experiments, WEIGHTS_DIR, CLASS_NAMES, IMG_SIZE, EPOCHS
)
from src.cnn.cnn import CNNScratch
from src.cnn.layers import (
    Conv2DLayer, MaxPooling2DLayer, AveragePooling2DLayer,
    GlobalAveragePooling2DLayer, FlattenLayer,
    LocallyConnected2DLayer, ActivationLayer
)
from src.shared.dense import DenseLayer
from src.shared.image_utils import load_image_cnn, load_batch_cnn, extract_features_cnn
from src.cnn.utils import (
    evaluate_keras_model, evaluate_scratch_model,
    plot_history, plot_f1_comparison
)

DATA_DIR = '../../data/intel'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print('Setup selesai')
print(f'Kelas: {CLASS_NAMES}')

## 1. Load Dataset

In [ ]:
train_ds, val_ds = load_dataset(DATA_DIR)

# kumpulkan test set ke numpy untuk evaluasi scratch
X_test, y_test = [], []
for X_batch, y_batch in val_ds:
    X_test.append(X_batch.numpy())
    y_test.append(y_batch.numpy())
X_test = np.concatenate(X_test)
y_test = np.concatenate(y_test)

print(f'Test set shape  : {X_test.shape}')
print(f'Label shape     : {y_test.shape}')
print(f'Distribusi label: {dict(zip(*np.unique(y_test, return_counts=True)))}')

In [ ]:
# tampilkan sample gambar per kelas
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for cls_idx, ax in enumerate(axes):
    mask = y_test == cls_idx
    img  = X_test[mask][0]
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[cls_idx])
    ax.axis('off')
plt.suptitle('Sample Gambar per Kelas')
plt.tight_layout()
plt.show()

## 2. Training Semua Variasi Conv2D (16 Arsitektur)
> **SKIP cell ini kalau bobot sudah ada di weights/cnn/**

In [ ]:
# variasi: 2 jumlah layer x 2 filter x 2 kernel x 2 pooling = 16 arsitektur
results = run_experiments(DATA_DIR)
print('Semua eksperimen selesai')

## 3. Definisi Konfigurasi

In [ ]:
filter_variants = {
    'small': {2: [32, 64],    3: [32, 64, 64]},
    'large': {2: [64, 128],   3: [64, 128, 128]}
}
kernel_variants = {
    'k3': {2: [3, 3],   3: [3, 3, 3]},
    'k5': {2: [5, 5],   3: [5, 5, 5]}
}

all_configs = []
for n_layers in [2, 3]:
    for fkey, f_dict in filter_variants.items():
        for kkey, k_dict in kernel_variants.items():
            for pool in ['max', 'average']:
                all_configs.append({
                    'name':            f'conv{n_layers}_{fkey}_{kkey}_{pool}',
                    'num_conv_layers': n_layers,
                    'filters':         f_dict[n_layers],
                    'kernel_sizes':    k_dict[n_layers],
                    'pooling_type':    pool
                })

print(f'Total konfigurasi: {len(all_configs)}')

## 4. Evaluasi Semua Variasi — Macro F1-Score

In [ ]:
all_results = {}

for cfg in all_configs:
    w_path = os.path.join(WEIGHTS_DIR, f"{cfg['name']}.weights.h5")
    if not os.path.exists(w_path):
        print(f"SKIP {cfg['name']} — bobot tidak ditemukan")
        continue

    model = build_cnn_model(
        cfg['num_conv_layers'], cfg['filters'],
        cfg['kernel_sizes'],    cfg['pooling_type']
    )
    model.load_weights(w_path)

    res = evaluate_keras_model(model, X_test, y_test, CLASS_NAMES)
    all_results[cfg['name']] = {**res, 'config': cfg}
    print(f"{cfg['name']:<35} macro F1 = {res['macro_f1']:.4f}")

print(f'\nTotal model dievaluasi: {len(all_results)}')

## 5. Analisis Pengaruh Jumlah Conv Layer

In [ ]:
# fix: large, k3, max — variasi jumlah layer
labels, scores = [], []
for n in [2, 3]:
    name = f'conv{n}_large_k3_max'
    labels.append(f'{n} layers')
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jumlah Conv Layer — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for n in [2, 3]:
    name = f'conv{n}_large_k3_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=f'{n} layers')
    axes[1].plot(h['val_loss'], label=f'{n} layers')

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jumlah Conv Layer')
plt.tight_layout(); plt.show()

## 6. Analisis Pengaruh Jumlah Filter

In [ ]:
filter_labels = {'small': '[32,64]', 'large': '[64,128]'}
labels, scores = [], []
for fkey in ['small', 'large']:
    name = f'conv2_{fkey}_k3_max'
    labels.append(filter_labels[fkey])
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jumlah Filter — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for fkey in ['small', 'large']:
    name = f'conv2_{fkey}_k3_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=filter_labels[fkey])
    axes[1].plot(h['val_loss'], label=filter_labels[fkey])

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jumlah Filter')
plt.tight_layout(); plt.show()

## 7. Analisis Pengaruh Ukuran Filter

In [ ]:
kernel_labels = {'k3': '3x3', 'k5': '5x5'}
labels, scores = [], []
for kkey in ['k3', 'k5']:
    name = f'conv2_large_{kkey}_max'
    labels.append(kernel_labels[kkey])
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Ukuran Filter — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for kkey in ['k3', 'k5']:
    name = f'conv2_large_{kkey}_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=kernel_labels[kkey])
    axes[1].plot(h['val_loss'], label=kernel_labels[kkey])

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Ukuran Filter')
plt.tight_layout(); plt.show()

## 8. Analisis Pengaruh Jenis Pooling

In [ ]:
labels, scores = [], []
for pool in ['max', 'average']:
    name = f'conv2_large_k3_{pool}'
    labels.append(pool)
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jenis Pooling — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for pool in ['max', 'average']:
    name = f'conv2_large_k3_{pool}'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=pool)
    axes[1].plot(h['val_loss'], label=pool)

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jenis Pooling')
plt.tight_layout(); plt.show()

## 9. Rekap Semua 16 Arsitektur & Pilih Terbaik

In [ ]:
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['macro_f1'], reverse=True)

print(f"{'Nama':<35} {'Macro F1':>10}")
print('-' * 47)
for name, res in sorted_results:
    print(f"{name:<35} {res['macro_f1']:>10.4f}")

BEST_NAME   = sorted_results[0][0]
BEST_CONFIG = sorted_results[0][1]['config']
print(f'\nArsitektur terbaik : {BEST_NAME}')
print(f'Filters            : {BEST_CONFIG["filters"]}')
print(f'Kernel sizes       : {BEST_CONFIG["kernel_sizes"]}')
print(f'Pooling            : {BEST_CONFIG["pooling_type"]}')
print(f'Num layers         : {BEST_CONFIG["num_conv_layers"]}')

In [ ]:
all_labels = [name for name, _ in sorted_results]
all_scores = [res['macro_f1'] for _, res in sorted_results]
plot_f1_comparison(all_labels, all_scores, title='Macro F1 — Semua 16 Arsitektur Conv2D')

## 10. Feature Extraction menggunakan Arsitektur Terbaik

In [ ]:
# kumpulkan semua path gambar train dan test
train_paths = glob.glob(
    os.path.join(DATA_DIR, 'seg_train', 'seg_train', '**', '*.jpg'),
    recursive=True
)
test_paths = glob.glob(
    os.path.join(DATA_DIR, 'seg_test', 'seg_test', '**', '*.jpg'),
    recursive=True
)

print(f'Train images: {len(train_paths)}')
print(f'Test images : {len(test_paths)}')

In [ ]:
# load arsitektur terbaik sebagai encoder (frozen)
encoder = build_cnn_model(
    BEST_CONFIG['num_conv_layers'],
    BEST_CONFIG['filters'],
    BEST_CONFIG['kernel_sizes'],
    BEST_CONFIG['pooling_type']
)
encoder.load_weights(os.path.join(WEIGHTS_DIR, f'{BEST_NAME}.weights.h5'))
encoder.trainable = False

# ekstraksi feature train
extract_features_cnn(
    paths       = train_paths,
    output_path = os.path.join(DATA_DIR, 'features_train.npy'),
    encoder     = encoder,
    batch_size  = 64,
    target_dim  = (150, 150)
)

# ekstraksi feature test
extract_features_cnn(
    paths       = test_paths,
    output_path = os.path.join(DATA_DIR, 'features_test.npy'),
    encoder     = encoder,
    batch_size  = 64,
    target_dim  = (150, 150)
)

In [ ]:
# verifikasi hasil feature extraction
features_train = np.load(
    os.path.join(DATA_DIR, 'features_train.npy'), allow_pickle=True
).item()
features_test  = np.load(
    os.path.join(DATA_DIR, 'features_test.npy'), allow_pickle=True
).item()

sample_key    = list(features_train.keys())[0]
print(f'Jumlah fitur train : {len(features_train)}')
print(f'Jumlah fitur test  : {len(features_test)}')
print(f'Shape feature vector: {features_train[sample_key].shape}')

## 11. Perbandingan Keras vs From Scratch (Shared Parameter / Conv2D)

In [ ]:
# load arsitektur terbaik
best_model = build_cnn_model(
    BEST_CONFIG['num_conv_layers'],
    BEST_CONFIG['filters'],
    BEST_CONFIG['kernel_sizes'],
    BEST_CONFIG['pooling_type']
)
best_model.load_weights(os.path.join(WEIGHTS_DIR, f'{BEST_NAME}.weights.h5'))
best_model.summary()

In [ ]:
# build scratch dari arsitektur terbaik
# kernel+bias dari Keras, activation dipisah pakai ActivationLayer
n_layers  = BEST_CONFIG['num_conv_layers']
best_pool = BEST_CONFIG['pooling_type']

scratch_layers = []
layer_idx = 1  # skip InputLayer di index 0

for i in range(n_layers):
    # Conv2D: padding='same', strides=(1,1), activation='relu'
    # sama persis dengan yang didefinisikan di build_cnn_model
    scratch_layers.append(
        Conv2DLayer(
            keras_layer = best_model.layers[layer_idx],
            activation  = 'relu',
            strides     = (1, 1),
            padding     = 'same'
        )
    )
    layer_idx += 1

    # Pooling: pool_size=(2,2) sama persis dengan build_cnn_model
    if best_pool == 'max':
        scratch_layers.append(MaxPooling2DLayer(pool_size=(2,2), strides=(2,2)))
    else:
        scratch_layers.append(AveragePooling2DLayer(pool_size=(2,2), strides=(2,2)))
    layer_idx += 1

# GlobalAveragePooling
scratch_layers.append(GlobalAveragePooling2DLayer())
layer_idx += 1

# Dense relu — aktivasi dipisah pakai ActivationLayer
scratch_layers.append(DenseLayer(keras_layer=best_model.layers[layer_idx]))
scratch_layers.append(ActivationLayer('relu'))
layer_idx += 1

# Dropout — skip
layer_idx += 1

# Dense softmax — aktivasi dipisah pakai ActivationLayer
scratch_layers.append(DenseLayer(keras_layer=best_model.layers[layer_idx]))
scratch_layers.append(ActivationLayer('softmax'))

scratch_model = CNNScratch(scratch_layers)
print(f'Jumlah layer scratch: {len(scratch_model.layers)}')

In [ ]:
# evaluasi Keras vs Scratch — pakai subset karena scratch lambat
N_EVAL = 200

keras_res   = evaluate_keras_model(best_model,     X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)
scratch_res = evaluate_scratch_model(scratch_model, X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)

print(f'Keras   macro F1 : {keras_res["macro_f1"]:.4f}')
print(f'Scratch macro F1 : {scratch_res["macro_f1"]:.4f}')
print(f'Selisih          : {abs(keras_res["macro_f1"] - scratch_res["macro_f1"]):.6f}')
print(f'Prediksi berbeda : {np.sum(keras_res["y_pred"] != scratch_res["y_pred"])} dari {N_EVAL}')

In [ ]:
plot_f1_comparison(
    ['Keras', 'From Scratch'],
    [keras_res['macro_f1'], scratch_res['macro_f1']],
    title='Conv2D — Keras vs From Scratch'
)
print('=== Keras ===')
print(keras_res['report'])
print('=== From Scratch ===')
print(scratch_res['report'])

## 12. Training LocallyConnected2D (Non-Shared Parameter)
> **SKIP kalau sudah pernah ditraining**

In [ ]:
# training LocallyConnected dengan config arsitektur terbaik
lc_model = build_locally_connected_model(
    filters      = BEST_CONFIG['filters'],
    kernel_sizes = BEST_CONFIG['kernel_sizes'],
    pooling_type = BEST_CONFIG['pooling_type']
)
lc_model.summary()

lc_history = lc_model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, verbose=1
)

lc_w_path = os.path.join(WEIGHTS_DIR, 'locally_connected_best.weights.h5')
lc_model.save_weights(lc_w_path)
print(f'Bobot LC disimpan: {lc_w_path}')

## 13. Perbandingan Shared vs Non-Shared

In [ ]:
# load LC kalau sudah disimpan
lc_model = build_locally_connected_model(
    filters      = BEST_CONFIG['filters'],
    kernel_sizes = BEST_CONFIG['kernel_sizes'],
    pooling_type = BEST_CONFIG['pooling_type']
)
lc_model.load_weights(os.path.join(WEIGHTS_DIR, 'locally_connected_best.weights.h5'))

# evaluasi
shared_res    = evaluate_keras_model(best_model, X_test, y_test, CLASS_NAMES)
nonshared_res = evaluate_keras_model(lc_model,   X_test, y_test, CLASS_NAMES)

print(f'Conv2D (shared)               F1: {shared_res["macro_f1"]:.4f}')
print(f'LocallyConnected (non-shared) F1: {nonshared_res["macro_f1"]:.4f}')
print(f'\nParameter Conv2D           : {best_model.count_params():,}')
print(f'Parameter LocallyConnected  : {lc_model.count_params():,}')

In [ ]:
# plot loss shared vs non-shared
best_history = results[BEST_NAME]['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(best_history['loss'],           label='Conv2D (shared)')
axes[0].plot(lc_history.history['loss'],     label='LocallyConnected')
axes[1].plot(best_history['val_loss'],        label='Conv2D (shared)')
axes[1].plot(lc_history.history['val_loss'], label='LocallyConnected')

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Shared vs Non-Shared')
plt.tight_layout(); plt.show()

plot_f1_comparison(
    ['Conv2D\n(shared)', 'LocallyConnected\n(non-shared)'],
    [shared_res['macro_f1'], nonshared_res['macro_f1']],
    title='Shared vs Non-Shared — Macro F1'
)

## 14. From Scratch — LocallyConnected2D

In [ ]:
# build scratch LocallyConnected
# LocallyConnected di Keras default padding='valid'
lc_scratch_layers = []
layer_idx = 1

for i in range(len(BEST_CONFIG['filters'])):
    lc_scratch_layers.append(
        LocallyConnected2DLayer(
            keras_layer = lc_model.layers[layer_idx],
            activation  = 'relu',
            strides     = (1, 1),
            padding     = 'valid',
            kernel_size = (BEST_CONFIG['kernel_sizes'][i], BEST_CONFIG['kernel_sizes'][i])
        )
    )
    layer_idx += 1

    if BEST_CONFIG['pooling_type'] == 'max':
        lc_scratch_layers.append(MaxPooling2DLayer(pool_size=(2,2), strides=(2,2)))
    else:
        lc_scratch_layers.append(AveragePooling2DLayer(pool_size=(2,2), strides=(2,2)))
    layer_idx += 1

# Flatten
lc_scratch_layers.append(FlattenLayer())
layer_idx += 1

# Dense relu — aktivasi dipisah
lc_scratch_layers.append(DenseLayer(keras_layer=lc_model.layers[layer_idx]))
lc_scratch_layers.append(ActivationLayer('relu'))
layer_idx += 1

# Dropout — skip
layer_idx += 1

# Dense softmax — aktivasi dipisah
lc_scratch_layers.append(DenseLayer(keras_layer=lc_model.layers[layer_idx]))
lc_scratch_layers.append(ActivationLayer('softmax'))

lc_scratch = CNNScratch(lc_scratch_layers)
print(f'Jumlah layer LC scratch: {len(lc_scratch.layers)}')

In [ ]:
lc_keras_res   = evaluate_keras_model(lc_model,    X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)
lc_scratch_res = evaluate_scratch_model(lc_scratch, X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)

print(f'LC Keras   macro F1 : {lc_keras_res["macro_f1"]:.4f}')
print(f'LC Scratch macro F1 : {lc_scratch_res["macro_f1"]:.4f}')
print(f'Selisih             : {abs(lc_keras_res["macro_f1"] - lc_scratch_res["macro_f1"]):.6f}')
print(f'Prediksi berbeda    : {np.sum(lc_keras_res["y_pred"] != lc_scratch_res["y_pred"])} dari {N_EVAL}')

## 15. Rekap Akhir Semua Perbandingan

In [ ]:
print('='*55)
print(f"{'Model':<40} {'Macro F1':>10}")
print('='*55)
print(f"{'Conv2D terbaik (Keras)':<40} {shared_res['macro_f1']:>10.4f}")
print(f"{'Conv2D terbaik (Scratch)':<40} {scratch_res['macro_f1']:>10.4f}")
print(f"{'LocallyConnected (Keras)':<40} {nonshared_res['macro_f1']:>10.4f}")
print(f"{'LocallyConnected (Scratch)':<40} {lc_scratch_res['macro_f1']:>10.4f}")
print('='*55)
print(f"\nParameter Conv2D           : {best_model.count_params():,}")
print(f"Parameter LocallyConnected  : {lc_model.count_params():,}")

In [ ]:
plot_f1_comparison(
    ['Conv2D\nKeras', 'Conv2D\nScratch', 'LC\nKeras', 'LC\nScratch'],
    [
        shared_res['macro_f1'],
        scratch_res['macro_f1'],
        nonshared_res['macro_f1'],
        lc_scratch_res['macro_f1']
    ],
    title='Rekap Akhir — Macro F1 Semua Model'
)